This notebook ingests and prepares data from the Office for National Statistics (ONS) Birth Characteristics dataset (2022), which provides detailed information on births by factors such as gestational age, birthweight, ethnicity, and geography across England and Wales. The purpose is to construct a clean, standardised national dataset capturing ethnicity-specific birth outcomes, forming the foundation for understanding disparities in maternal and neonatal health.

In [2]:
from pathlib import Path
import importlib.util
import re
import pandas as pd

HERE = Path.cwd()
PARQUET_ENGINE = "pyarrow" if importlib.util.find_spec("pyarrow") else "fastparquet"

RAW_PATH = HERE / "raw_birthcharacteristics2022.xlsx"
ETH_MAP_PATH = HERE / "_config_ethnicity_mapping.csv"

OUT_PATH = HERE / "_processed_ons_birth_characteristics_2022_national.parquet"
QA_PATH = HERE / "_qa_ons_birth_characteristics_2022_ingestion_issues.csv"

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Missing file: {RAW_PATH.name}")

In [3]:
# Load ethnicity mapping 
if not ETH_MAP_PATH.exists():
    raise FileNotFoundError(f"Missing file: {ETH_MAP_PATH.name}")

eth_map = pd.read_csv(ETH_MAP_PATH)

required_map_cols = {"raw_ethnicity", "standard_ethnicity"}
missing_map = required_map_cols - set(eth_map.columns)
if missing_map:
    raise KeyError(f"Ethnicity mapping missing columns: {sorted(missing_map)}")

ETH_MAP = dict(
    zip(
        eth_map["raw_ethnicity"].astype(str).str.strip(),
        eth_map["standard_ethnicity"].astype(str).str.strip(),
    )
)

In [4]:
# Enumerate sheets and find candidate tables 
xls = pd.ExcelFile(RAW_PATH)
sheet_names = xls.sheet_names
sheet_names[:20], len(sheet_names)

(['Cover_sheet',
  'Contents',
  'Notes',
  'Table_1',
  'Table_2',
  'Table_3',
  'Table_4',
  'Table_5',
  'Table_6',
  'Table_7',
  'Table_8',
  'Table_9',
  'Table_10',
  'Table_11',
  'Table_12',
  'Table_13',
  'Table_14',
  'Table_15',
  'Table_16',
  'Table_17'],
 23)

In [5]:
# scan tables for ethnicity + key outcome signals

target_sheets = [s for s in sheet_names if s.startswith("Table_")]

summary = []
for s in target_sheets:
    # read with a few header offsets; stop at first that looks tabular
    df = None
    for header in [0, 1, 2, 3, 4, 5, 6, 7, 8]:
        try:
            tmp = pd.read_excel(RAW_PATH, sheet_name=s, header=header)
            tmp.columns = [str(c).strip() for c in tmp.columns]
            tmp = tmp.dropna(axis=1, how="all")
            if tmp.shape[0] > 10 and tmp.shape[1] > 2:
                df = tmp
                break
        except Exception:
            continue

    if df is None:
        summary.append({"sheet": s, "status": "unreadable"})
        continue

    cols_lower = [c.lower() for c in df.columns]
    has_eth = any("ethnic" in c for c in cols_lower)

    # look for likely outcome keywords in column names
    kw = {
        "gestation": any("gest" in c for c in cols_lower),
        "birthweight": any("weight" in c or "birthweight" in c for c in cols_lower),
        "stillbirth": any("still" in c for c in cols_lower),
        "neonatal": any("neonat" in c for c in cols_lower),
        "low_birthweight": any("low" in c and "weight" in c for c in cols_lower),
    }

    # also scan first 30 rows (stringified) for these keywords because ONS sometimes stores as rows
    sample_text = " ".join(df.head(30).astype(str).fillna("").values.flatten()).lower()
    kw_rows = {
        "gestation_row": "gest" in sample_text,
        "birthweight_row": "birthweight" in sample_text or "weight" in sample_text,
        "stillbirth_row": "still" in sample_text,
        "neonatal_row": "neonat" in sample_text,
    }

    summary.append(
        {
            "sheet": s,
            "rows": df.shape[0],
            "cols": df.shape[1],
            "has_ethnicity_col": has_eth,
            **kw,
            **kw_rows,
            "columns_preview": df.columns[:8].tolist(),
        }
    )

pd.DataFrame(summary).sort_values(
    ["has_ethnicity_col", "gestation", "birthweight", "stillbirth", "neonatal"],
    ascending=False
).head(30)

,sheet,rows,cols,has_ethnicity_col,gestation,birthweight,stillbirth,neonatal,low_birthweight,gestation_row,birthweight_row,stillbirth_row,neonatal_row,columns_preview
6,Table_7,33,15,True,True,False,False,False,False,True,True,True,False,[Worksheet 7: Births by gestational age at bir...
18,Table_19,22,31,True,False,False,True,False,False,False,False,True,False,[Worksheet 19: Live births and stillbirths (nu...
19,Table_20,22,40,True,False,False,True,False,False,False,False,True,False,[Worksheet 20: Live births and stillbirths (nu...
17,Table_18,104,9,False,True,True,True,False,False,True,True,False,False,[Worksheet 18: Stillbirths by ONS cause groups...
5,Table_6,382,5,False,True,True,False,False,True,True,True,False,False,[Worksheet 6: Live births of 37 weeks or more ...
8,Table_9,19,10,False,True,False,True,False,False,True,False,False,False,[Worksheet 9: Stillbirths by gestational age a...
7,Table_8,19,12,False,True,False,False,False,False,True,True,False,False,[Worksheet 8: Live births by gestational age a...
9,Table_10,15,17,False,True,False,False,False,False,True,True,False,False,[Worksheet 10: Live births by number of previo...
4,Table_5,21,23,False,False,True,True,False,False,False,True,False,False,[Worksheet 5: Stillbirths (numbers and rates) ...
3,Table_4,19,15,False,False,True,False,False,False,False,True,False,False,[Worksheet 4: Live births by birthweight and a...


In [6]:
# Table_7: strip notes + promote the true header row so it's analysis-ready

def norm_text(x: object) -> str:
    return re.sub(r"\s+", " ", str(x)).strip().lower()

# Read raw sheet with no header
t7_raw = pd.read_excel(RAW_PATH, sheet_name="Table_7", header=None)

# Find the row that contains the real header (gestational age label)
header_idx = None
for i in range(0, 80):
    row0 = norm_text(t7_raw.iloc[i, 0])
    if "gestational age at birth" in row0 and "weeks" in row0:
        header_idx = i
        break

if header_idx is None:
    raise ValueError("Table_7: could not locate header row containing 'Gestational age at birth (weeks)'")

# Promote that row to header; keep rows below
t7 = t7_raw.copy()
t7.columns = t7.iloc[header_idx].tolist()
t7 = t7.iloc[header_idx + 1:].reset_index(drop=True)

# Drop completely empty columns/rows
t7 = t7.dropna(axis=1, how="all").dropna(axis=0, how="all").copy()

# Clean column names (remove extra whitespace, keep newlines if present)
t7.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in t7.columns]

# Ensure the key first column is present
gest_col = None
for c in t7.columns:
    c_norm = norm_text(c)
    if "gestational age at birth" in c_norm and "weeks" in c_norm:
        gest_col = c
        break

if gest_col is None:
    raise KeyError("Table_7: expected gestational age column not found after header promotion")

# Trim the gestation category text
t7[gest_col] = t7[gest_col].astype(str).str.strip()

# Remove any trailing footnote-type rows (e.g., 'Percentage not stated') if they exist
# Keep rows until we hit the first completely non-gestation label block (conservative)
drop_labels = {"percentage not stated"}
t7 = t7.loc[~t7[gest_col].str.lower().isin(drop_labels)].copy()

print("Table_7 header row index:", header_idx)
print("Gestation column:", repr(gest_col))
print("Shape:", t7.shape)
print("Columns:", t7.columns.tolist())

display(t7.head(15))

Table_7 header row index: 7
Gestation column: 'Gestational age at birth (weeks)'
Shape: (25, 15)
Columns: ['Gestational age at birth (weeks)', 'Total live births', 'Total stillbirths', 'Live births Bangladeshi', 'Live births Indian', 'Live births Pakistani', 'Live births Any other Asian background', 'Live births Black African', 'Live births Black Caribbean', 'Live births Any other Black background', 'Live births Mixed/multiple', 'Live births Any other ethnic group', 'Live births White British', 'Live births White Other', 'Live births Not stated']


,Gestational age at birth (weeks),Total live births,Total stillbirths,Live births Bangladeshi,Live births Indian,Live births Pakistani,Live births Any other Asian background,Live births Black African,Live births Black Caribbean,Live births Any other Black background,Live births Mixed/multiple,Live births Any other ethnic group,Live births White British,Live births White Other,Live births Not stated
0,All gestational ages,605060,2408,10353,24607,27723,17484,24188,5222,3995,42245,14917,345460,67875,20991
1,"Under 22 weeks and birthweight less than 1,000g",305,0,6,17,26,6,32,7,3,25,5,140,26,12
2,22 weeks,229,0,4,10,26,11,21,5,4,23,7,90,13,15
3,23 weeks,306,0,8,9,19,11,25,6,8,17,9,153,17,24
4,24 weeks,405,217,8,26,30,10,36,10,7,28,17,180,33,20
5,25 weeks,385,218,7,15,29,16,33,7,10,26,14,173,31,24
6,26 weeks,558,176,20,24,29,17,39,12,5,43,18,288,42,21
7,27 weeks,648,162,15,27,39,20,47,12,8,44,21,322,61,32
8,28 weeks,796,128,17,31,34,34,34,8,8,59,22,441,65,43
9,29 weeks,985,107,10,47,48,32,54,22,8,56,27,547,84,50


In [7]:
# Finalise Table_7 into clean long engineering dataset

import numpy as np
import re

def to_numeric_clean(x: object) -> float:
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(",", "")
    if s.lower() in {"", "nan", ":", "-", "x", "[low]"}:
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan


# --- Convert numeric columns ---
numeric_cols = [c for c in t7.columns if c != gest_col]

for c in numeric_cols:
    t7[c] = t7[c].map(to_numeric_clean)


# --- Reshape to long ---
t7_long = t7.melt(
    id_vars=[gest_col],
    value_vars=numeric_cols,
    var_name="measure_raw",
    value_name="value"
)

t7_long = t7_long.rename(columns={
    gest_col: "category"
})


# --- Derive metric type ---
def classify_metric(col_name: str) -> str:
    if col_name == "Total live births":
        return "total_live_births"
    if col_name == "Total stillbirths":
        return "total_stillbirths"
    if col_name.startswith("Live births"):
        return "live_births"
    return "unknown"


t7_long["metric"] = t7_long["measure_raw"].map(classify_metric)


# --- Extract ethnicity ---
def extract_ethnicity(col_name: str) -> str:
    if col_name in {"Total live births", "Total stillbirths"}:
        return "All"

    match = re.match(r"Live births (.*)", col_name)
    if match:
        return match.group(1).strip()

    return col_name


t7_long["ethnicity_raw"] = t7_long["measure_raw"].map(extract_ethnicity)

# Standardise using your config mapping dictionary
t7_long["ethnicity_standard"] = t7_long["ethnicity_raw"].map(
    lambda x: ETH_MAP.get(x, x)
)


# --- Add context ---
t7_long["table"] = "Table_7"
t7_long["source"] = "ONS Birth Characteristics"
t7_long["year"] = 2022
t7_long["geography"] = "England and Wales"


# Remove rows where value is entirely missing
t7_long = t7_long.loc[t7_long["value"].notna()].copy()


print("Final Table_7 long shape:", t7_long.shape)
display(t7_long.head(10))


Final Table_7 long shape: (350, 10)


,category,measure_raw,value,metric,ethnicity_raw,ethnicity_standard,table,source,year,geography
0,All gestational ages,Total live births,605060.0,total_live_births,All,All,Table_7,ONS Birth Characteristics,2022,England and Wales
1,"Under 22 weeks and birthweight less than 1,000g",Total live births,305.0,total_live_births,All,All,Table_7,ONS Birth Characteristics,2022,England and Wales
2,22 weeks,Total live births,229.0,total_live_births,All,All,Table_7,ONS Birth Characteristics,2022,England and Wales
3,23 weeks,Total live births,306.0,total_live_births,All,All,Table_7,ONS Birth Characteristics,2022,England and Wales
4,24 weeks,Total live births,405.0,total_live_births,All,All,Table_7,ONS Birth Characteristics,2022,England and Wales
5,25 weeks,Total live births,385.0,total_live_births,All,All,Table_7,ONS Birth Characteristics,2022,England and Wales
6,26 weeks,Total live births,558.0,total_live_births,All,All,Table_7,ONS Birth Characteristics,2022,England and Wales
7,27 weeks,Total live births,648.0,total_live_births,All,All,Table_7,ONS Birth Characteristics,2022,England and Wales
8,28 weeks,Total live births,796.0,total_live_births,All,All,Table_7,ONS Birth Characteristics,2022,England and Wales
9,29 weeks,Total live births,985.0,total_live_births,All,All,Table_7,ONS Birth Characteristics,2022,England and Wales


In [8]:
t7_long["ethnicity_raw"].unique()

<ArrowStringArray>
[                       'All',                'Bangladeshi',
                     'Indian',                  'Pakistani',
 'Any other Asian background',              'Black African',
            'Black Caribbean', 'Any other Black background',
             'Mixed/multiple',     'Any other ethnic group',
              'White British',                'White Other',
                 'Not stated']
Length: 13, dtype: str

In [9]:
t7_long[t7_long["ethnicity_raw"].str.contains("Black", na=False)].head(10)

,category,measure_raw,value,metric,ethnicity_raw,ethnicity_standard,table,source,year,geography
150,All gestational ages,Live births Black African,24188.0,live_births,Black African,Black African,Table_7,ONS Birth Characteristics,2022,England and Wales
151,"Under 22 weeks and birthweight less than 1,000g",Live births Black African,32.0,live_births,Black African,Black African,Table_7,ONS Birth Characteristics,2022,England and Wales
152,22 weeks,Live births Black African,21.0,live_births,Black African,Black African,Table_7,ONS Birth Characteristics,2022,England and Wales
153,23 weeks,Live births Black African,25.0,live_births,Black African,Black African,Table_7,ONS Birth Characteristics,2022,England and Wales
154,24 weeks,Live births Black African,36.0,live_births,Black African,Black African,Table_7,ONS Birth Characteristics,2022,England and Wales
155,25 weeks,Live births Black African,33.0,live_births,Black African,Black African,Table_7,ONS Birth Characteristics,2022,England and Wales
156,26 weeks,Live births Black African,39.0,live_births,Black African,Black African,Table_7,ONS Birth Characteristics,2022,England and Wales
157,27 weeks,Live births Black African,47.0,live_births,Black African,Black African,Table_7,ONS Birth Characteristics,2022,England and Wales
158,28 weeks,Live births Black African,34.0,live_births,Black African,Black African,Table_7,ONS Birth Characteristics,2022,England and Wales
159,29 weeks,Live births Black African,54.0,live_births,Black African,Black African,Table_7,ONS Birth Characteristics,2022,England and Wales


In [10]:
# Cell 7 (replace again) — Table_19: robust header detection (handles NaN/float cells)

import pandas as pd
import numpy as np
import re


# Read raw without header
t19_raw = pd.read_excel(RAW_PATH, sheet_name="Table_19", header=None)

# Find the header row by scanning for both phrases in ANY cell on the same row
header_idx = None
for i in range(0, min(120, len(t19_raw))):
    row_values = ["" if pd.isna(v) else str(v).lower() for v in t19_raw.iloc[i].tolist()]
    row_join = " | ".join(row_values)

    if ("area of usual residence" in row_join) and ("code" in row_join):
        header_idx = i
        break

if header_idx is None:
    raise ValueError("Could not detect true header row in Table_19")

# Promote header
t19 = t19_raw.copy()
t19.columns = t19.iloc[header_idx].tolist()
t19 = t19.iloc[header_idx + 1:].reset_index(drop=True)

# Drop empty columns/rows
t19 = t19.dropna(axis=1, how="all").dropna(axis=0, how="all").copy()

# Clean column names
t19.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in t19.columns]

print("Table_19 true header row index:", header_idx)
print("Shape:", t19.shape)
print("Columns (first 12):", t19.columns[:12].tolist())

display(t19.head(10))

Table_19 true header row index: 9
Shape: (13, 29)
Columns (first 12): ['Area of usual residence Code', 'Area of usual residence Name', 'Area of usual residence Geography', 'Number of live births Total', 'Number of live births Asian', 'Number of live births Black', 'Number of live births Mixed/multiple', 'Number of live births Any other ethnic group', 'Number of live births White', 'Number of live births Not stated', 'Number of stillbirths Total', 'Number of stillbirths Asian']


,Area of usual residence Code,Area of usual residence Name,Area of usual residence Geography,Number of live births Total,Number of live births Asian,Number of live births Black,Number of live births Mixed/multiple,Number of live births Any other ethnic group,Number of live births White,Number of live births Not stated,...,Unreliable indicator Stillbirth rate Asian,Stillbirth rate Black,Unreliable indicator Stillbirth rate Black,Stillbirth rate Mixed/multiple,Unreliable indicator Stillbirth rate Mixed/multiple,Stillbirth rate Any other ethnic group,Unreliable indicator Stillbirth rate Any other ethnic group,Stillbirth rate White,Stillbirth rate Not stated,Unreliable indicator Stillbirth rate Not stated
0,"K04000001, J99000001","ENGLAND, WALES AND ELSEWHERE",Country,605060,80167,33405,42245,14917,413335,20991,...,NaN,6.5,NaN,4,NaN,4.5,NaN,3.5,5.5,NaN
1,K04000001,ENGLAND AND WALES,Country,604923,80160,33399,42236,14907,413244,20977,...,NaN,6.5,NaN,4,NaN,4.4,NaN,3.5,5.5,NaN
2,E92000001,ENGLAND,Country,576643,79317,32975,41504,14568,396752,11527,...,NaN,6.5,NaN,4,NaN,4.3,NaN,3.4,6.7,NaN
3,E12000001,NORTH EAST,Region,24676,1425,697,897,542,21079,36,...,[u],7.1,[u],4.4,[u],[x],[u],4,[x],[u]
4,E12000002,NORTH WEST,Region,76334,10165,3088,4139,2010,54759,2173,...,NaN,7.1,NaN,6.2,NaN,4.5,[u],3.5,5.5,[u]
5,E12000003,YORKSHIRE AND THE HUMBER,Region,55168,7659,2201,2671,1481,39631,1525,...,NaN,5.4,[u],6.3,[u],3.4,[u],3.4,6.5,[u]
6,E12000004,EAST MIDLANDS,Region,46837,5040,1899,3067,662,35134,1035,...,NaN,3.7,[u],3.6,[u],7.5,[u],3.3,10.5,[u]
7,E12000005,WEST MIDLANDS,Region,63344,12678,4298,4858,1339,39555,616,...,NaN,7.2,NaN,5.7,NaN,2.2,[u],3.6,6.5,[u]
8,E12000006,EAST,Region,64270,6074,2553,4388,1048,47034,3173,...,NaN,8.5,NaN,2.7,[u],4.7,[u],3.8,5,[u]
9,E12000007,LONDON,Region,106597,25140,13762,11984,5376,48712,1623,...,NaN,6.4,NaN,4.2,NaN,5,NaN,2.9,6.1,[u]


In [11]:
def to_numeric_clean(x: object) -> float:
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(",", "")
    if s.lower() in {"", "nan", ":", "-", "x", "[low]", "[u]", "u"}:
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan


# --- 1) Keep national-level rows only (Country geography) ---
# This avoids region-level duplication in later EDA unless you explicitly want it.
t19_national = t19.loc[t19["Area of usual residence Geography"].astype(str).str.strip() == "Country"].copy()

print("Table_19 national rows:", t19_national.shape[0])
display(t19_national[["Area of usual residence Code", "Area of usual residence Name", "Area of usual residence Geography"]].head(10))


# --- 2) Numeric conversion for measure columns ---
id_cols = ["Area of usual residence Code", "Area of usual residence Name", "Area of usual residence Geography"]
measure_cols = [c for c in t19_national.columns if c not in id_cols]

for c in measure_cols:
    t19_national[c] = t19_national[c].map(to_numeric_clean)


# --- 3) Melt to long ---
t19_long = t19_national.melt(
    id_vars=id_cols,
    value_vars=measure_cols,
    var_name="measure_raw",
    value_name="value"
)

t19_long = t19_long.loc[t19_long["value"].notna()].copy()


# --- 4) Parse metric + ethnicity from measure_raw ---
def parse_metric_and_ethnicity(measure: str) -> tuple[str, str]:
    m = str(measure).strip()

    # Remove "Unreliable indicator ..." columns from ingestion (they are flags, not measures)
    if m.lower().startswith("unreliable indicator"):
        return "unreliable_flag", "All"

    # Expected patterns:
    # "Number of live births Black"
    # "Number of stillbirths White"
    # "Stillbirth rate Mixed/multiple"
    parts = m.split(" ")

    if m.startswith("Number of live births"):
        eth = m.replace("Number of live births", "").strip()
        return "number_live_births", eth if eth else "All"

    if m.startswith("Number of stillbirths"):
        eth = m.replace("Number of stillbirths", "").strip()
        return "number_stillbirths", eth if eth else "All"

    if m.startswith("Stillbirth rate"):
        eth = m.replace("Stillbirth rate", "").strip()
        return "stillbirth_rate", eth if eth else "All"

    # fallback
    return "unknown", "All"


parsed = t19_long["measure_raw"].map(parse_metric_and_ethnicity)
t19_long["metric"], t19_long["ethnicity_raw"] = zip(*parsed)

# Drop unreliable flags from the main long dataset (we can keep separately later if needed)
t19_long = t19_long.loc[t19_long["metric"] != "unreliable_flag"].copy()

# Standardise ethnicity using mapping dictionary
t19_long["ethnicity_standard"] = t19_long["ethnicity_raw"].map(lambda x: ETH_MAP.get(x, x))

# Context fields
t19_long["table"] = "Table_19"
t19_long["source"] = "ONS Birth Characteristics"
t19_long["year"] = 2022
t19_long["geography"] = "England and Wales"  # analytic context; the row still carries 'Country'

print("Table_19 long shape:", t19_long.shape)
display(t19_long.head(10))

Table_19 national rows: 4


,Area of usual residence Code,Area of usual residence Name,Area of usual residence Geography
0,"K04000001, J99000001","ENGLAND, WALES AND ELSEWHERE",Country
1,K04000001,ENGLAND AND WALES,Country
2,E92000001,ENGLAND,Country
12,W92000004,WALES,Country


Table_19 long shape: (83, 12)


,Area of usual residence Code,Area of usual residence Name,Area of usual residence Geography,measure_raw,value,metric,ethnicity_raw,ethnicity_standard,table,source,year,geography
0,"K04000001, J99000001","ENGLAND, WALES AND ELSEWHERE",Country,Number of live births Total,605060.0,number_live_births,Total,All,Table_19,ONS Birth Characteristics,2022,England and Wales
1,K04000001,ENGLAND AND WALES,Country,Number of live births Total,604923.0,number_live_births,Total,All,Table_19,ONS Birth Characteristics,2022,England and Wales
2,E92000001,ENGLAND,Country,Number of live births Total,576643.0,number_live_births,Total,All,Table_19,ONS Birth Characteristics,2022,England and Wales
3,W92000004,WALES,Country,Number of live births Total,28280.0,number_live_births,Total,All,Table_19,ONS Birth Characteristics,2022,England and Wales
4,"K04000001, J99000001","ENGLAND, WALES AND ELSEWHERE",Country,Number of live births Asian,80167.0,number_live_births,Asian,Asian,Table_19,ONS Birth Characteristics,2022,England and Wales
5,K04000001,ENGLAND AND WALES,Country,Number of live births Asian,80160.0,number_live_births,Asian,Asian,Table_19,ONS Birth Characteristics,2022,England and Wales
6,E92000001,ENGLAND,Country,Number of live births Asian,79317.0,number_live_births,Asian,Asian,Table_19,ONS Birth Characteristics,2022,England and Wales
7,W92000004,WALES,Country,Number of live births Asian,843.0,number_live_births,Asian,Asian,Table_19,ONS Birth Characteristics,2022,England and Wales
8,"K04000001, J99000001","ENGLAND, WALES AND ELSEWHERE",Country,Number of live births Black,33405.0,number_live_births,Black,Black,Table_19,ONS Birth Characteristics,2022,England and Wales
9,K04000001,ENGLAND AND WALES,Country,Number of live births Black,33399.0,number_live_births,Black,Black,Table_19,ONS Birth Characteristics,2022,England and Wales


In [12]:
# Finalise Table_19 to a single national record (England and Wales) + align schema with Table_7

# Keep only "ENGLAND AND WALES"
t19_ew = t19_long.loc[
    t19_long["Area of usual residence Name"].astype(str).str.strip() == "ENGLAND AND WALES"
].copy()

if t19_ew.empty:
    raise ValueError("Could not find the 'ENGLAND AND WALES' row in Table_19 after filtering.")

# Rename id fields to consistent names
t19_ew = t19_ew.rename(columns={
    "Area of usual residence Code": "area_code",
    "Area of usual residence Name": "area_name",
    "Area of usual residence Geography": "area_geography",
})

# Add a consistent "category" field (Table_19 isn't categorical like gestation; use a fixed label)
t19_ew["category"] = "All"

# Keep a consistent set of columns to simplify later concatenation / EDA
t19_ew_out = t19_ew[
    [
        "table",
        "source",
        "year",
        "geography",
        "area_geography",
        "area_code",
        "area_name",
        "category",
        "metric",
        "ethnicity_raw",
        "ethnicity_standard",
        "measure_raw",
        "value",
    ]
].copy()

print("Table_19 England and Wales shape:", t19_ew_out.shape)
display(t19_ew_out.head(10))

Table_19 England and Wales shape: (21, 13)


,table,source,year,geography,area_geography,area_code,area_name,category,metric,ethnicity_raw,ethnicity_standard,measure_raw,value
1,Table_19,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,All,number_live_births,Total,All,Number of live births Total,604923.0
5,Table_19,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,All,number_live_births,Asian,Asian,Number of live births Asian,80160.0
9,Table_19,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,All,number_live_births,Black,Black,Number of live births Black,33399.0
13,Table_19,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,All,number_live_births,Mixed/multiple,Mixed/multiple,Number of live births Mixed/multiple,42236.0
17,Table_19,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,All,number_live_births,Any other ethnic group,Any other ethnic group,Number of live births Any other ethnic group,14907.0
21,Table_19,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,All,number_live_births,White,White,Number of live births White,413244.0
25,Table_19,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,All,number_live_births,Not stated,Not stated,Number of live births Not stated,20977.0
29,Table_19,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,All,number_stillbirths,Total,All,Number of stillbirths Total,2379.0
33,Table_19,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,All,number_stillbirths,Asian,Asian,Number of stillbirths Asian,375.0
37,Table_19,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,All,number_stillbirths,Black,Black,Number of stillbirths Black,217.0


In [13]:
# Cell 10 (replace) — Ingest Table_20 (time series by year × detailed ethnicity): promote true header + preview

import pandas as pd
import numpy as np
import re


# Read raw without header
t20_raw = pd.read_excel(RAW_PATH, sheet_name="Table_20", header=None)

# Detect header row by finding a row that contains "Year" and "Number of live births"
t20_header_idx = None
for i in range(0, min(120, len(t20_raw))):
    row_values = ["" if pd.isna(v) else str(v).strip().lower() for v in t20_raw.iloc[i].tolist()]
    row_join = " | ".join(row_values)
    if ("year" in row_join) and ("number of live births" in row_join):
        t20_header_idx = i
        break

if t20_header_idx is None:
    raise ValueError("Could not detect true header row in Table_20 (looking for 'Year' + 'Number of live births').")

# Promote header
t20 = t20_raw.copy()
t20.columns = t20.iloc[t20_header_idx].tolist()
t20 = t20.iloc[t20_header_idx + 1:].reset_index(drop=True)

# Drop empty columns/rows
t20 = t20.dropna(axis=1, how="all").dropna(axis=0, how="all").copy()

# Clean column names (flatten newlines)
t20.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in t20.columns]

print("Table_20 true header row index:", t20_header_idx)
print("Shape:", t20.shape)
print("Columns (first 12):", t20.columns[:12].tolist())

display(t20.head(10))

Table_20 true header row index: 6
Shape: (16, 40)
Columns (first 12): ['Year', 'Number of live births Total', 'Number of live births Bangladeshi', 'Number of live births Indian', 'Number of live births Pakistani', 'Number of live births Any other Asian background', 'Number of live births Black African', 'Number of live births Black Caribbean', 'Number of live births Any other Black background', 'Number of live births Mixed/ multiple', 'Number of live births Any other ethnic group', 'Number of live births White British']


,Year,Number of live births Total,Number of live births Bangladeshi,Number of live births Indian,Number of live births Pakistani,Number of live births Any other Asian background,Number of live births Black African,Number of live births Black Caribbean,Number of live births Any other Black background,Number of live births Mixed/ multiple,...,Stillbirth rates Pakistani,Stillbirth rates Any other Asian background,Stillbirth rates Black African,Stillbirth rates Black Caribbean,Stillbirth rates Any other Black background,Stillbirth rates Mixed/ multiple,Stillbirth rates Any other ethnic group,Stillbirth rates White British,Stillbirth rates Any other White background,Stillbirth rates Not stated
0,2022,605060,10353,24607,27723,17484,24188,5222,3995,42245,...,5.1,4.7,6.9,4.6,6,4,4.5,3.5,3.7,5.5
1,2021,624162,10158,22896,26950,15919,20983,5079,3804,42928,...,5.9,4.3,7,6.6,6.8,4.4,4.5,3.5,4,7.1
2,2020,613231,9614,21435,27397,16207,21098,5064,3720,40751,...,5.6,5.2,6.2,6.1,7.2,3.5,3.9,3.1,3.9,7.2
3,2019,639987,9505,20627,27573,16169,21589,5480,3777,41918,...,5.9,4.7,7,5.8,9.7,3,3.8,3.3,3.9,5.9
4,2018,656723,9684,20143,26994,15966,21823,5540,3958,41291,...,5.8,4.4,7.6,5.7,7.3,3.9,5.5,3.5,3.5,5.9
5,2017,678728,10089,21157,28135,16664,22734,5718,4113,41381,...,6.6,5.5,7,7.3,8,4,4.5,3.6,4.1,4.8
6,2016,695427,10450,21700,28682,17807,23635,5933,4168,41931,...,6.9,4.5,7.8,7,6.9,4.6,5.8,3.7,4.2,6.6
7,2015,696365,9889,21582,28142,17363,23483,5964,4014,40433,...,7.2,4.5,8.2,8.6,7.9,4.6,4.6,3.8,4.3,6.4
8,2014,694610,9764,22217,28645,16251,23176,5990,4375,37450,...,7.5,4,7.5,7,7.3,4.7,4.5,4.1,4.3,5.9
9,2013,698130,10063,22673,29099,15411,23723,6525,4534,36046,...,8.4,3.9,7.2,8.5,7.4,4.7,5.4,4,3.9,7.2


In [14]:
# Table_20: reshape to long + standardise ethnicity (no derived metrics)

def to_numeric_clean(x: object) -> float:
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(",", "")
    if s.lower() in {"", "nan", ":", "-", "x", "[low]", "[u]", "u"}:
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan


# Ensure Year is numeric
t20["Year"] = pd.to_numeric(t20["Year"], errors="coerce")
t20 = t20.loc[t20["Year"].notna()].copy()
t20["Year"] = t20["Year"].astype(int)

id_cols = ["Year"]
measure_cols = [c for c in t20.columns if c not in id_cols]

# Convert measures to numeric
for c in measure_cols:
    t20[c] = t20[c].map(to_numeric_clean)

# Melt
t20_long = t20.melt(
    id_vars=id_cols,
    value_vars=measure_cols,
    var_name="measure_raw",
    value_name="value",
)

t20_long = t20_long.loc[t20_long["value"].notna()].copy()

# Parse metric + ethnicity
def parse_metric_and_ethnicity(measure: str) -> tuple[str, str]:
    m = str(measure).strip()

    if m.lower().startswith("unreliable indicator"):
        return "unreliable_flag", "All"

    # "Number of live births Black African"
    if m.startswith("Number of live births"):
        eth = m.replace("Number of live births", "").strip()
        return "number_live_births", eth if eth else "All"

    # "Number of stillbirths Black African"
    if m.startswith("Number of stillbirths"):
        eth = m.replace("Number of stillbirths", "").strip()
        return "number_stillbirths", eth if eth else "All"

    # "Stillbirth rates Black African" (note plural in this sheet)
    if m.startswith("Stillbirth rates"):
        eth = m.replace("Stillbirth rates", "").strip()
        return "stillbirth_rate", eth if eth else "All"

    # fallback
    return "unknown", "All"


parsed = t20_long["measure_raw"].map(parse_metric_and_ethnicity)
t20_long["metric"], t20_long["ethnicity_raw"] = zip(*parsed)

# Drop unreliable flags (optional to keep separately later)
t20_long = t20_long.loc[t20_long["metric"] != "unreliable_flag"].copy()

# Normalise ethnicity_raw spacing quirks (e.g., "Mixed/ multiple")
t20_long["ethnicity_raw"] = (
    t20_long["ethnicity_raw"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.replace("Mixed/ multiple", "Mixed/multiple", regex=False)
    .str.strip()
)

# Standardise ethnicity
t20_long["ethnicity_standard"] = t20_long["ethnicity_raw"].map(lambda x: ETH_MAP.get(x, x))

# Add consistent fields
t20_long["table"] = "Table_20"
t20_long["source"] = "ONS Birth Characteristics"
t20_long["geography"] = "England and Wales"
t20_long = t20_long.rename(columns={"Year": "year"})
t20_long["category"] = "All"

print("Table_20 long shape:", t20_long.shape)
display(t20_long.head(20))

Table_20 long shape: (624, 10)


,year,measure_raw,value,metric,ethnicity_raw,ethnicity_standard,table,source,geography,category
0,2022,Number of live births Total,605060.0,number_live_births,Total,All,Table_20,ONS Birth Characteristics,England and Wales,All
1,2021,Number of live births Total,624162.0,number_live_births,Total,All,Table_20,ONS Birth Characteristics,England and Wales,All
2,2020,Number of live births Total,613231.0,number_live_births,Total,All,Table_20,ONS Birth Characteristics,England and Wales,All
3,2019,Number of live births Total,639987.0,number_live_births,Total,All,Table_20,ONS Birth Characteristics,England and Wales,All
4,2018,Number of live births Total,656723.0,number_live_births,Total,All,Table_20,ONS Birth Characteristics,England and Wales,All
5,2017,Number of live births Total,678728.0,number_live_births,Total,All,Table_20,ONS Birth Characteristics,England and Wales,All
6,2016,Number of live births Total,695427.0,number_live_births,Total,All,Table_20,ONS Birth Characteristics,England and Wales,All
7,2015,Number of live births Total,696365.0,number_live_births,Total,All,Table_20,ONS Birth Characteristics,England and Wales,All
8,2014,Number of live births Total,694610.0,number_live_births,Total,All,Table_20,ONS Birth Characteristics,England and Wales,All
9,2013,Number of live births Total,698130.0,number_live_births,Total,All,Table_20,ONS Birth Characteristics,England and Wales,All


In [15]:
# Cell — Verify all sheets containing "ethnic" in header rows

import pandas as pd

xls = pd.ExcelFile(RAW_PATH)

ethnicity_sheets = []

for sheet in xls.sheet_names:
    df_preview = pd.read_excel(RAW_PATH, sheet_name=sheet, header=None, nrows=40)

    found = False
    for i in range(len(df_preview)):
        row_vals = ["" if pd.isna(v) else str(v).lower() for v in df_preview.iloc[i]]
        if any("ethnic" in v for v in row_vals):
            found = True
            break

    if found:
        ethnicity_sheets.append(sheet)

print("Sheets containing 'ethnic' in header preview:")
print(ethnicity_sheets)

Sheets containing 'ethnic' in header preview:
['Cover_sheet', 'Contents', 'Table_7', 'Table_19', 'Table_20']


In [16]:
# Cell — Robust scan for ethnicity-related content across all sheets

import pandas as pd

xls = pd.ExcelFile(RAW_PATH)

ethnicity_keywords = [
    "black",
    "white",
    "asian",
    "mixed",
    "bangladeshi",
    "caribbean",
    "african",
]

ethnicity_sheets = []

for sheet in xls.sheet_names:
    df_preview = pd.read_excel(RAW_PATH, sheet_name=sheet, header=None, nrows=50)

    found = False
    for i in range(len(df_preview)):
        row_vals = ["" if pd.isna(v) else str(v).lower() for v in df_preview.iloc[i]]
        row_join = " | ".join(row_vals)

        if any(keyword in row_join for keyword in ethnicity_keywords):
            found = True
            break

    if found:
        ethnicity_sheets.append(sheet)

print("Sheets containing ethnicity-related keywords:")
print(ethnicity_sheets)

Sheets containing ethnicity-related keywords:
['Cover_sheet', 'Table_1', 'Table_6', 'Table_7', 'Table_19', 'Table_20']


In [17]:
# Cell — Confirm whether additional sheets actually contain ethnicity columns

import pandas as pd

candidate_sheets = ['Table_1', 'Table_6']

for sheet in candidate_sheets:
    df_preview = pd.read_excel(RAW_PATH, sheet_name=sheet, header=None, nrows=40)

    header_like_rows = []
    
    for i in range(len(df_preview)):
        row_vals = ["" if pd.isna(v) else str(v).lower() for v in df_preview.iloc[i]]
        
        # If row contains multiple ethnicity category words in same row, likely a header
        ethnicity_terms = ["black", "white", "asian", "mixed", "bangladeshi", "caribbean", "african"]
        count = sum(term in " ".join(row_vals) for term in ethnicity_terms)
        
        if count >= 2:
            header_like_rows.append((sheet, i, df_preview.iloc[i].tolist()))
    
    print(f"\nSheet: {sheet}")
    if header_like_rows:
        print("Potential ethnicity header rows found:")
        for row in header_like_rows[:2]:
            print("Row index:", row[1])
            print(row[2])
    else:
        print("No ethnicity-structured header rows detected.")


Sheet: Table_1
No ethnicity-structured header rows detected.

Sheet: Table_6
No ethnicity-structured header rows detected.


In [18]:
# Harmonise Table_7 / Table_19 / Table_20 schemas + write processed parquet + QA mapping check

from pathlib import Path
import pandas as pd

# ---- 1) Align Table_7 long to the common schema ----
# t7_long currently has: category (gestation), metric, ethnicity_raw/standard, year, geography, table, source, measure_raw, value

t7_out = t7_long.copy()

# Ensure expected columns exist
required_t7 = {"table", "source", "year", "geography", "category", "metric", "ethnicity_raw", "ethnicity_standard", "measure_raw", "value"}
missing_t7 = required_t7 - set(t7_out.columns)
if missing_t7:
    raise KeyError(f"Table_7 missing expected columns: {sorted(missing_t7)}")

# Add area fields (not applicable for Table_7)
t7_out["area_geography"] = "Country"
t7_out["area_code"] = "K04000001"
t7_out["area_name"] = "ENGLAND AND WALES"

# ---- 2) Align Table_19 England & Wales output ----
t19_out = t19_ew_out.copy()

# Standardise year/geography already present
t19_out["category"] = t19_out["category"].astype(str)
# Ensure columns exist
required_t19 = {"table", "source", "year", "geography", "area_geography", "area_code", "area_name", "category",
               "metric", "ethnicity_raw", "ethnicity_standard", "measure_raw", "value"}
missing_t19 = required_t19 - set(t19_out.columns)
if missing_t19:
    raise KeyError(f"Table_19 missing expected columns: {sorted(missing_t19)}")

# ---- 3) Align Table_20 long ----
t20_out = t20_long.copy()

# Add area fields for consistency (Table_20 is England & Wales time series)
t20_out["area_geography"] = "Country"
t20_out["area_code"] = "K04000001"
t20_out["area_name"] = "ENGLAND AND WALES"

required_t20 = {"table", "source", "year", "geography", "category", "metric", "ethnicity_raw", "ethnicity_standard", "measure_raw", "value",
               "area_geography", "area_code", "area_name"}
missing_t20 = required_t20 - set(t20_out.columns)
if missing_t20:
    raise KeyError(f"Table_20 missing expected columns: {sorted(missing_t20)}")

# ---- 4) Concatenate all ONS birth characteristics (ethnicity-related) ----
common_cols = [
    "table", "source", "year", "geography",
    "area_geography", "area_code", "area_name",
    "category", "metric",
    "ethnicity_raw", "ethnicity_standard",
    "measure_raw", "value",
]

ons_birth_characteristics = pd.concat(
    [t7_out[common_cols], t19_out[common_cols], t20_out[common_cols]],
    ignore_index=True
)

print("Combined ONS birth characteristics shape:", ons_birth_characteristics.shape)
display(ons_birth_characteristics.head(10))

# ---- 5) QA: unmapped ethnicity labels ----
# ETH_MAP keys are the raw labels we recognise
eth_map_keys = set(ETH_MAP.keys())
unmapped = (
    ons_birth_characteristics[["ethnicity_raw", "ethnicity_standard"]]
    .drop_duplicates()
    .loc[lambda d: ~d["ethnicity_raw"].isin(eth_map_keys) & (d["ethnicity_standard"] == d["ethnicity_raw"])]
    .sort_values("ethnicity_raw")
)

qa_path = Path("qa_ons_birth_characteristics_unmapped_ethnicity.csv")
if not unmapped.empty:
    unmapped.to_csv(qa_path, index=False)
    print(f"Wrote QA ethnicity mapping issues: {qa_path}")
else:
    print("No unmapped ethnicity labels detected (or all are intentionally passthrough).")


# ---- 6) Write processed parquet (canonical) ----
output_path = Path("_processed_ons_birth_characteristics_national.parquet")
ons_birth_characteristics.to_parquet(output_path, index=False)
print(f"Wrote processed dataset: {output_path}")
print("Shape:", ons_birth_characteristics.shape)

Combined ONS birth characteristics shape: (995, 13)


,table,source,year,geography,area_geography,area_code,area_name,category,metric,ethnicity_raw,ethnicity_standard,measure_raw,value
0,Table_7,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,All gestational ages,total_live_births,All,All,Total live births,605060.0
1,Table_7,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,"Under 22 weeks and birthweight less than 1,000g",total_live_births,All,All,Total live births,305.0
2,Table_7,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,22 weeks,total_live_births,All,All,Total live births,229.0
3,Table_7,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,23 weeks,total_live_births,All,All,Total live births,306.0
4,Table_7,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,24 weeks,total_live_births,All,All,Total live births,405.0
5,Table_7,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,25 weeks,total_live_births,All,All,Total live births,385.0
6,Table_7,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,26 weeks,total_live_births,All,All,Total live births,558.0
7,Table_7,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,27 weeks,total_live_births,All,All,Total live births,648.0
8,Table_7,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,28 weeks,total_live_births,All,All,Total live births,796.0
9,Table_7,ONS Birth Characteristics,2022,England and Wales,Country,K04000001,ENGLAND AND WALES,29 weeks,total_live_births,All,All,Total live births,985.0


Wrote QA ethnicity mapping issues: qa_ons_birth_characteristics_unmapped_ethnicity.csv
Wrote processed dataset: _processed_ons_birth_characteristics_national.parquet
Shape: (995, 13)


In [19]:
# Structural integrity checks before closing Notebook 08

print("Total rows:", ons_birth_characteristics.shape[0])

print("\nTables present:")
print(ons_birth_characteristics["table"].value_counts())

print("\nMetrics present:")
print(ons_birth_characteristics["metric"].value_counts())

print("\nYears present:")
print(sorted(ons_birth_characteristics["year"].unique()))

print("\nEthnicity_standard unique values:")
print(sorted(ons_birth_characteristics["ethnicity_standard"].unique()))

Total rows: 995

Tables present:
table
Table_20    624
Table_7     350
Table_19     21
Name: count, dtype: int64

Metrics present:
metric
live_births           300
number_live_births    215
number_stillbirths    215
stillbirth_rate       215
total_live_births      25
total_stillbirths      25
Name: count, dtype: int64

Years present:
[np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]

Ethnicity_standard unique values:
['All', 'Any other Asian background', 'Any other Black background', 'Any other White background', 'Any other ethnic group', 'Asian', 'Bangladeshi', 'Black', 'Black African', 'Black Caribbean', 'Indian', 'Mixed/multiple', 'Not stated', 'Pakistani', 'White', 'White British', 'White Other']


The ONS birth characteristics data has been cleaned, restructured, and standardised across relevant tables to produce a consistent, analysis-ready dataset of ethnicity-related outcomes. Complex spreadsheet formats have been resolved, key variables harmonised, and outputs validated for completeness and integrity. The resulting dataset provides a robust national baseline for analysing health inequalities and will be used in combination with local data sources in subsequent stages.